# LeetCode 32 — Longest Valid Parentheses (Hard) bằng Python

Notebook này giảng **từ căn bản** bài toán dấu ngoặc nổi tiếng của LeetCode. Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestValidParentheses(self, s: str) -> int:

```

---

## Sau notebook này, bạn sẽ nắm được

1. Thế nào là một chuỗi ngoặc **hợp lệ** (valid / well-formed) và cách kiểm tra bằng một **biến đếm** (balance).
2. Cách **brute force** (thử mọi đoạn con) và vì sao nó quá chậm với `n = 30 000`.
3. **Quy hoạch động** (Dynamic Programming — DP) với ý tưởng "đáp án *kết thúc tại* vị trí `i`".
4. **Stack lưu index với mốc `-1`** — lời giải đẹp nhất, sẽ được trace thật kỹ.
5. **Hai lượt quét** trái→phải và phải→trái chỉ dùng 2 biến đếm (bộ nhớ `O(1)`), và *vì sao bắt buộc cần lượt ngược*.
6. Lỗi hay gặp, mẹo nhận dạng pattern, các bài liên quan và 6 bài tập có đáp án.

Lộ trình: `Brute force O(n³)` → `Brute force O(n²)` → `DP O(n)` → `Stack O(n)` → `Hai lượt quét O(n), O(1) nhớ`.

## 1. Đề bài

Cho một chuỗi `s` **chỉ gồm** các ký tự `'('` và `')'`. Hãy trả về **độ dài** của **chuỗi con liên tiếp** (substring) **hợp lệ** dài nhất.

> Gốc tiếng Anh: *Given a string containing just the characters `'('` and `')'`, return the length of the longest valid (well-formed) parentheses substring.*

### Ví dụ trong đề

| Input `s` | Output | Giải thích |
|---|:---:|---|
| `"(()"` | `2` | Chuỗi con hợp lệ dài nhất là `"()"` |
| `")()())"` | `4` | Chuỗi con hợp lệ dài nhất là `"()()"` |
| `""` | `0` | Chuỗi rỗng, không có gì |

### Constraints (ràng buộc)

- `0 <= s.length <= 3 * 10^4`
- `s[i]` là `'('` hoặc `')'`.

Chú ý: đề hỏi **độ dài** (một số nguyên), không hỏi chính chuỗi con.

## 2. Hiểu đúng hai chữ "hợp lệ" và "chuỗi con"

**Hợp lệ** (well-formed) nghĩa là mỗi `'('` được đóng bởi đúng một `')'` phía sau nó, và các cặp lồng nhau đúng thứ tự. Ví dụ:

| Chuỗi | Hợp lệ? | Vì sao |
|---|:---:|---|
| `"()"` | ✔ | một cặp |
| `"(())"` | ✔ | cặp lồng trong cặp |
| `"()()"` | ✔ | hai cặp đứng cạnh nhau |
| `")("` | ✘ | `')'` xuất hiện khi chưa có `'('` nào để đóng |
| `"(()"` | ✘ | thừa một `'('` chưa đóng |

**Chuỗi con (substring)** là một đoạn **liên tiếp** `s[i:j]`. Khác với **dãy con (subsequence)** được phép bỏ ký tự ở giữa.
Ví dụ `s = "()(()"`: nếu được bỏ ký tự thì lấy được `"()()"` (độ dài 4), nhưng đây **không** phải đoạn liên tiếp. Đáp án đúng là `2`.

Trong Python, `s[i:j]` lấy các ký tự từ index `i` đến `j - 1` (không gồm `j`), nên độ dài là `j - i`.

In [1]:
s = "()(()"
print("s[0:2] =", repr(s[0:2]), "| độ dài", 2 - 0)
print("s[2:5] =", repr(s[2:5]), "| độ dài", 5 - 2)
print("s[3:5] =", repr(s[3:5]), "| độ dài", 5 - 3)
print("Số ký tự trong s:", len(s))

s[0:2] = '()' | độ dài 2
s[2:5] = '(()' | độ dài 3
s[3:5] = '()' | độ dài 2
Số ký tự trong s: 5


## 3. Kiến thức nền 1 — Kiểm tra chuỗi ngoặc hợp lệ bằng một biến đếm

Vì chỉ có **một loại** ngoặc, ta không cần stack để kiểm tra hợp lệ. Chỉ cần một biến `balance` ("số `'('` đang mở chưa đóng"):

- gặp `'('` → `balance += 1`
- gặp `')'` → `balance -= 1`

Chuỗi hợp lệ **khi và chỉ khi** hai điều kiện cùng đúng:

1. **Trong lúc duyệt**, `balance` **không bao giờ âm** (âm nghĩa là có `')'` đóng mà chẳng có gì để đóng).
2. **Kết thúc**, `balance == 0` (không còn `'('` nào bị bỏ lơ).

Ví dụ `"(())"`: balance đi `1 → 2 → 1 → 0` ✔. Ví dụ `"())("`: `1 → 0 → -1` ✘ dừng ngay.

In [2]:
def is_valid(t: str) -> bool:
    balance = 0
    for c in t:
        balance += 1 if c == "(" else -1
        if balance < 0:          # có ')' thừa -> hỏng ngay
            return False
    return balance == 0          # không được thừa '('

for t in ["()", "(())", "()()", ")(", "(()", "())(", ""]:
    print(f"{t!r:8} -> {is_valid(t)}")

'()'     -> True
'(())'   -> True
'()()'   -> True
')('     -> False
'(()'    -> False
'())('   -> False
''       -> True


### Trace biến `balance`

Hàm `trace_balance` in giá trị `balance` sau từng ký tự để thấy rõ thời điểm chuỗi "hỏng". Ta thử một chuỗi hợp lệ, một chuỗi bị âm giữa chừng và một chuỗi thừa `'('` ở cuối.

In [3]:
def trace_balance(t: str) -> None:
    balance = 0
    steps = []
    for c in t:
        balance += 1 if c == "(" else -1
        steps.append(f"{c}:{balance}")
        if balance < 0:
            print(f"{t!r:10} | {' '.join(steps)}  -> âm, KHÔNG hợp lệ")
            return
    verdict = "hợp lệ" if balance == 0 else f"thừa {balance} '(' , KHÔNG hợp lệ"
    print(f"{t!r:10} | {' '.join(steps)}  -> {verdict}")

trace_balance("(()())")
trace_balance("())(")
trace_balance("((())")

'(()())'   | (:1 (:2 ):1 (:2 ):1 ):0  -> hợp lệ
'())('     | (:1 ):0 ):-1  -> âm, KHÔNG hợp lệ
'((())'    | (:1 (:2 (:3 ):2 ):1  -> thừa 1 '(' , KHÔNG hợp lệ


### Kiến thức nền 2 — Liệt kê mọi chuỗi con

Một chuỗi hợp lệ luôn có **độ dài chẵn** (mỗi `'('` đi với đúng một `')'`). Vậy để liệt kê ứng viên, ta chọn điểm đầu `i` và điểm cuối `j` sao cho `j - i` chẵn:

```python
for i in range(n):
    for j in range(i + 2, n + 1, 2):   # bước nhảy 2 -> chỉ độ dài chẵn
        candidate = s[i:j]
```

`range(start, stop, step)` đếm từ `start`, cộng thêm `step` mỗi lần, dừng **trước** `stop`.

In [4]:
print(list(range(2, 9, 2)))       # [2, 4, 6, 8]

s = ")()())"
n = len(s)
for i in range(n):
    subs = [s[i:j] for j in range(i + 2, n + 1, 2)]
    print(f"i={i}:", subs)

[2, 4, 6, 8]
i=0: [')(', ')()(', ')()())']
i=1: ['()', '()()']
i=2: [')(', ')())']
i=3: ['()']
i=4: ['))']
i=5: []


## 4. Cách 1 — Brute force: thử mọi chuỗi con độ dài chẵn

Ý tưởng tự nhiên nhất: duyệt **mọi** đoạn `s[i:j]` có độ dài chẵn, đoạn nào `is_valid` thì cập nhật đáp án bằng `max`.

`max(a, b)` trả về số lớn hơn; ta dùng `best = max(best, độ_dài)` để "giữ kỷ lục".

Độ phức tạp (gọi `n = len(s)`):

- Có khoảng `n² / 4` đoạn độ dài chẵn, mỗi đoạn kiểm tra mất tới `O(n)` → **Time `O(n³)`**.
- Mỗi lần cắt `s[i:j]` tạo chuỗi mới → **Space `O(n)`**.

In [5]:
def longest_valid_bruteforce(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i + 2, n + 1, 2):
            if is_valid(s[i:j]):
                best = max(best, j - i)
    return best

for s in ["(()", ")()())", "", "()(()", "()(())"]:
    print(f"{s!r:10} -> {longest_valid_bruteforce(s)}")

'(()'      -> 2
')()())'   -> 4
''         -> 0
'()(()'    -> 2
'()(())'   -> 6


### Cải tiến nhỏ: brute force `O(n²)` — cố định điểm đầu, kéo dài điểm cuối

**Cách cũ vướng ở đâu?** Với cùng một `i`, ta kiểm tra `s[i:i+2]`, rồi `s[i:i+4]`, ... mỗi lần **đếm lại từ đầu**. Thật lãng phí: `balance` của `s[i:j+1]` chỉ là `balance` của `s[i:j]` cộng thêm một ký tự!

**Mấu chốt:** giữ `balance` chạy liên tục khi kéo dài `j`:

- Nếu `balance < 0` → mọi đoạn bắt đầu tại `i` dài hơn cũng hỏng → `break` sớm.
- Nếu `balance == 0` → `s[i..j]` hợp lệ, độ dài `j - i + 1`.

In [6]:
def longest_valid_n2(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        balance = 0
        for j in range(i, n):
            balance += 1 if s[j] == "(" else -1
            if balance < 0:
                break                      # đoạn từ i đã hỏng, không cứu được
            if balance == 0:
                best = max(best, j - i + 1)
    return best

for s in ["(()", ")()())", "", "()(()", "()(())"]:
    print(f"{s!r:10} -> {longest_valid_n2(s)}")

'(()'      -> 2
')()())'   -> 4
''         -> 0
'()(()'    -> 2
'()(())'   -> 6


### Vì sao `O(n²)` vẫn chưa đủ?

Trường hợp xấu nhất là chuỗi toàn `'('`: `balance` không bao giờ âm nên không `break` được, vòng trong chạy hết. Cell dưới đếm số bước vòng trong: khi `n` gấp đôi, số bước **gấp bốn**.

Với `n = 3 * 10^4`, số bước ≈ `n² / 2 = 4.5 * 10^8` — Python cần hàng chục giây, chắc chắn **Time Limit Exceeded**. Ta cần một lời giải **một lượt `O(n)`**.

In [7]:
def count_inner_steps(s: str) -> int:
    steps = 0
    n = len(s)
    for i in range(n):
        balance = 0
        for j in range(i, n):
            steps += 1
            balance += 1 if s[j] == "(" else -1
            if balance < 0:
                break
    return steps

for n in [500, 1000, 2000]:
    print(f"n={n:5} -> {count_inner_steps('(' * n):>9,} bước")
print(f"Ước lượng n=30000 -> {30000 * 30001 // 2:,} bước")

n=  500 ->   125,250 bước
n= 1000 ->   500,500 bước
n= 2000 -> 2,001,000 bước
Ước lượng n=30000 -> 450,015,000 bước


## 5. Cách 2 — Quy hoạch động (DP): "đoạn hợp lệ dài nhất **kết thúc tại** `i`"

### Kiến thức nền: DP là gì?

**Dynamic Programming** = chia bài toán lớn thành các bài toán nhỏ **chồng lên nhau**, lưu đáp án nhỏ vào một bảng (thường là list `dp`) để dùng lại thay vì tính lại.

Một kiểu DP rất hay gặp với chuỗi/mảng: định nghĩa `dp[i]` = đáp án của đoạn **kết thúc đúng tại vị trí `i`**. Khi đó đáp án cuối cùng là `max(dp)`.

**Ví dụ mini:** tìm dãy chữ `'a'` liên tiếp dài nhất trong `"aabaaab"`.
Đặt `run[i]` = độ dài dãy `'a'` liên tiếp **kết thúc tại** `i`:

- nếu `t[i] == 'a'` thì `run[i] = run[i-1] + 1` (nối dài đoạn ngay trước),
- ngược lại `run[i] = 0`.

In [8]:
t = "aabaaab"
run = [0] * len(t)          # [0] * n tạo list n số 0
for i, c in enumerate(t):   # enumerate cho cả index i và ký tự c
    if c == "a":
        run[i] = (run[i - 1] if i > 0 else 0) + 1
print("t   =", list(t))
print("run =", run)
print("đáp án =", max(run))

t   = ['a', 'a', 'b', 'a', 'a', 'a', 'b']
run = [1, 2, 0, 1, 2, 3, 0]
đáp án = 3


### Định nghĩa DP cho bài ngoặc

> `dp[i]` = độ dài chuỗi con hợp lệ **dài nhất kết thúc đúng tại index `i`**.

- Nếu `s[i] == '('` → **không** chuỗi hợp lệ nào kết thúc bằng `'('` → `dp[i] = 0`.
- Nếu `s[i] == ')'`, xét ký tự ngay trước `s[i-1]`, có **hai trường hợp**:

**Trường hợp A — `s[i-1] == '('`** (dạng `...()`):
cặp `"()"` mới vừa đóng, nối thêm đoạn hợp lệ kết thúc tại `i-2`:

```text
dp[i] = dp[i-2] + 2          (dp[i-2] coi là 0 nếu i-2 < 0)
```

**Trường hợp B — `s[i-1] == ')'`** (dạng `...))`):
`s[i-1]` đã là đuôi của một đoạn hợp lệ dài `dp[i-1]`. Ký tự `')'` tại `i` phải ghép với ký tự **ngay trước đoạn đó**:

```text
j = i - dp[i-1] - 1          # vị trí "đối tác" tiềm năng của s[i]
nếu j >= 0 và s[j] == '(':
    dp[i] = dp[i-1] + 2 + dp[j-1]   (dp[j-1] coi là 0 nếu j-1 < 0)
```

Phần `+ dp[j-1]` là để **ghép thêm** đoạn hợp lệ nằm sát bên trái `j` — rất dễ quên!

### Tính tay trường hợp B với `s = "()(())"`

```text
index:  0 1 2 3 4 5
s    :  ( ) ( ( ) )
dp   :  0 2 0 0 2 ?
```

Tại `i = 5`: `s[4] == ')'` → trường hợp B. `dp[4] = 2` (đoạn `"()"` ở 3..4).
`j = 5 - 2 - 1 = 2`, `s[2] == '('` ✔ → ghép được: phần lõi `s[2..5] = "(())"` dài `dp[4] + 2 = 4`.
Bên trái `j` còn đoạn hợp lệ kết thúc tại `j - 1 = 1` dài `dp[1] = 2` → `dp[5] = 2 + 2 + 2 = 6`. Cả chuỗi hợp lệ!

In [9]:
s = "()(())"
dp = [0, 2, 0, 0, 2, None]   # đã tính tay tới i=4
i = 5
j = i - dp[i - 1] - 1
print("j =", j, "| s[j] =", repr(s[j]))
print("lõi    =", s[j:i + 1], "dài", dp[i - 1] + 2)
print("ghép thêm dp[j-1] =", dp[j - 1])
print("dp[5]  =", dp[i - 1] + 2 + dp[j - 1])

j = 2 | s[j] = '('
lõi    = (()) dài 4
ghép thêm dp[j-1] = 2
dp[5]  = 6


### Code DP hoàn chỉnh

In [10]:
def longest_valid_dp(s: str) -> int:
    n = len(s)
    dp = [0] * n
    best = 0
    for i in range(1, n):                 # i = 0 luôn có dp = 0
        if s[i] == "(":
            continue                      # dp[i] = 0
        if s[i - 1] == "(":               # Trường hợp A: ...()
            dp[i] = (dp[i - 2] if i >= 2 else 0) + 2
        else:                             # Trường hợp B: ...))
            j = i - dp[i - 1] - 1
            if j >= 0 and s[j] == "(":
                dp[i] = dp[i - 1] + 2 + (dp[j - 1] if j >= 1 else 0)
        best = max(best, dp[i])
    return best

for s in ["(()", ")()())", "", "()(()", "()(())"]:
    print(f"{s!r:10} -> {longest_valid_dp(s)}")

'(()'      -> 2
')()())'   -> 4
''         -> 0
'()(()'    -> 2
'()(())'   -> 6


### Trace DP với `s = "()(()))"`

Input này đi qua **đủ mọi nhánh**: ký tự `'('`, trường hợp A, trường hợp B ghép được (có cộng `dp[j-1]`), và trường hợp B thất bại vì `j < 0`.

| i | s[i] | s[i-1] | Nhánh | j | dp[i] |
|:-:|:-:|:-:|---|:-:|:-:|
| 0 | `(` | – | bỏ qua | – | 0 |
| 1 | `)` | `(` | A: `dp[-1]→0 + 2` | – | 2 |
| 2 | `(` | | `'('` → 0 | – | 0 |
| 3 | `(` | | `'('` → 0 | – | 0 |
| 4 | `)` | `(` | A: `dp[2] + 2` | – | 2 |
| 5 | `)` | `)` | B: `s[2]=='('`, `dp[4]+2+dp[1]` | 2 | 6 |
| 6 | `)` | `)` | B: `j = 6-6-1 = -1 < 0` → hỏng | -1 | 0 |

Đáp án `max(dp) = 6` (đoạn `s[0:6] = "()(())"`).

In [11]:
def trace_dp(s: str) -> int:
    n = len(s)
    dp = [0] * n
    for i in range(n):
        if s[i] == "(" or i == 0:
            note = "'(' -> 0" if s[i] == "(" else "i=0 -> 0"
        elif s[i - 1] == "(":
            dp[i] = (dp[i - 2] if i >= 2 else 0) + 2
            note = "A: ...()"
        else:
            j = i - dp[i - 1] - 1
            if j >= 0 and s[j] == "(":
                dp[i] = dp[i - 1] + 2 + (dp[j - 1] if j >= 1 else 0)
                note = f"B: j={j}, s[j]='(' -> ghép"
            else:
                note = f"B: j={j} -> không ghép được"
        print(f"i={i} s[i]={s[i]} | {note:28} | dp={dp[:i + 1]}")
    print("Đáp án:", max(dp, default=0))
    return max(dp, default=0)

trace_dp("()(()))")

i=0 s[i]=( | '(' -> 0                     | dp=[0]
i=1 s[i]=) | A: ...()                     | dp=[0, 2]
i=2 s[i]=( | '(' -> 0                     | dp=[0, 2, 0]
i=3 s[i]=( | '(' -> 0                     | dp=[0, 2, 0, 0]
i=4 s[i]=) | A: ...()                     | dp=[0, 2, 0, 0, 2]
i=5 s[i]=) | B: j=2, s[j]='(' -> ghép     | dp=[0, 2, 0, 0, 2, 6]
i=6 s[i]=) | B: j=-1 -> không ghép được   | dp=[0, 2, 0, 0, 2, 6, 0]
Đáp án: 6


6

### Độ phức tạp DP

- **Time `O(n)`**: một vòng `for`, mỗi bước chỉ vài phép tính.
- **Space `O(n)`**: list `dp` dài `n`.

**Điểm yếu:** công thức trường hợp B nhiều chỉ số (`i - dp[i-1] - 1`, `dp[j-1]`), dễ sai off-by-one khi phỏng vấn. Có cách nào "tự nhiên" hơn để tìm đối tác của mỗi `')'` không? → **Stack**.

## 6. Cách 3 — Stack lưu index với mốc `-1` (lời giải chính)

### Kiến thức nền: Stack (ngăn xếp)

**Stack** hoạt động như chồng đĩa: đĩa đặt vào **sau cùng** được lấy ra **đầu tiên** — gọi là **LIFO** (*Last In, First Out*). Trong Python dùng luôn `list`:

| Thao tác | Code | Ý nghĩa |
|---|---|---|
| push | `stack.append(x)` | đặt `x` lên đỉnh |
| pop | `stack.pop()` | lấy ra (và xoá) phần tử ở đỉnh |
| peek | `stack[-1]` | xem đỉnh, không xoá |
| rỗng? | `not stack` | `True` nếu stack rỗng |

Cả `append` và `pop()` ở cuối list đều `O(1)`.

In [12]:
stack = []
stack.append(10)
stack.append(20)
stack.append(30)
print("sau 3 lần push:", stack, "| đỉnh =", stack[-1])
print("pop ra:", stack.pop())
print("còn lại:", stack, "| rỗng?", not stack)
stack.pop(); stack.pop()
print("pop thêm 2 lần:", stack, "| rỗng?", not stack)

sau 3 lần push: [10, 20, 30] | đỉnh = 30
pop ra: 30
còn lại: [10, 20] | rỗng? False
pop thêm 2 lần: [] | rỗng? True


### Từ "ghép ngoặc" đến "lưu index"

Cách kinh điển để ghép ngoặc: gặp `'('` thì push, gặp `')'` thì pop một `'('` ra ghép. Nhưng bài này cần **độ dài đoạn**, nên ta push **index** chứ không push ký tự — có index thì trừ nhau ra được độ dài.

**Ý tưởng then chốt — phần tử mốc (base):** đáy stack luôn giữ index của **"bức tường"** — vị trí cuối cùng mà một đoạn hợp lệ *không thể vượt qua*:

- Ban đầu chưa có tường nào → đặt mốc giả `-1` (tường tưởng tượng nằm trước index 0).
- Một `')'` không có ai ghép → nó là bức tường mới.

Thuật toán:

```text
stack = [-1]
với mỗi i, c:
    nếu c == '(' : push i
    nếu c == ')' : pop()                       # thử ghép với '(' ở đỉnh
                   nếu stack rỗng: push i       # vừa pop mất tường -> ')' này là tường mới
                   ngược lại: best = max(best, i - stack[-1])
```

**Vì sao `i - stack[-1]`?** Sau khi pop, đỉnh stack là index **ngay trước** đoạn hợp lệ dài nhất kết thúc tại `i` (hoặc là một `'('` chưa ghép, hoặc là bức tường). Đoạn đó là `s[stack[-1]+1 .. i]`, dài `i - stack[-1]`.

In [13]:
def longest_valid_stack(s: str) -> int:
    stack = [-1]                 # mốc: "bức tường" giả trước index 0
    best = 0
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        else:
            stack.pop()          # ghép với '(' ở đỉnh (hoặc pop mất bức tường)
            if not stack:
                stack.append(i)  # ')' không ai ghép -> thành bức tường mới
            else:
                best = max(best, i - stack[-1])
    return best

for s in ["(()", ")()())", "", "()(()", "()(())"]:
    print(f"{s!r:10} -> {longest_valid_stack(s)}")

'(()'      -> 2
')()())'   -> 4
''         -> 0
'()(()'    -> 2
'()(())'   -> 6


### Trace Stack với `s = "())((())"`

```text
index: 0 1 2 3 4 5 6 7
s    : ( ) ) ( ( ( ) )
```

Input này cho thấy: một `')'` thừa tại index 2 **thay bức tường**, và một `'('` chưa ghép tại index 3 cũng đóng vai trò ranh giới.

| i | c | Hành động | Stack sau bước | Độ dài `i - stack[-1]` | best |
|:-:|:-:|---|---|:-:|:-:|
| – | – | khởi tạo | `[-1]` | – | 0 |
| 0 | `(` | push 0 | `[-1, 0]` | – | 0 |
| 1 | `)` | pop 0, đỉnh = -1 | `[-1]` | 1 - (-1) = 2 | 2 |
| 2 | `)` | pop -1 → rỗng → push 2 (tường mới) | `[2]` | – | 2 |
| 3 | `(` | push 3 | `[2, 3]` | – | 2 |
| 4 | `(` | push 4 | `[2, 3, 4]` | – | 2 |
| 5 | `(` | push 5 | `[2, 3, 4, 5]` | – | 2 |
| 6 | `)` | pop 5, đỉnh = 4 | `[2, 3, 4]` | 6 - 4 = 2 | 2 |
| 7 | `)` | pop 4, đỉnh = 3 | `[2, 3]` | 7 - 3 = 4 | 4 |

Đáp án `4` là đoạn `s[4:8] = "(())"`. Index `3` còn kẹt lại trong stack: `'('` đó không bao giờ được đóng nên đoạn hợp lệ không thể bắt đầu trước nó.

In [14]:
def trace_stack(s: str) -> int:
    stack = [-1]
    best = 0
    print(f"khởi tạo: stack={stack}")
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
            print(f"i={i} '(' " + f"push {i}".ljust(26) + f"| stack={stack}")
        else:
            top = stack.pop()
            if not stack:
                stack.append(i)
                print(f"i={i} ')' " + f"pop {top} -> rỗng, push {i}".ljust(26) + f"| stack={stack}  (tường mới)")
            else:
                length = i - stack[-1]
                best = max(best, length)
                print(f"i={i} ')' " + f"pop {top}, đỉnh={stack[-1]}".ljust(26) + f"| stack={stack}  len={length} best={best}")
    print("Đáp án:", best)
    return best

trace_stack("())((())")

khởi tạo: stack=[-1]
i=0 '(' push 0                    | stack=[-1, 0]
i=1 ')' pop 0, đỉnh=-1            | stack=[-1]  len=2 best=2
i=2 ')' pop -1 -> rỗng, push 2    | stack=[2]  (tường mới)
i=3 '(' push 3                    | stack=[2, 3]
i=4 '(' push 4                    | stack=[2, 3, 4]
i=5 '(' push 5                    | stack=[2, 3, 4, 5]
i=6 ')' pop 5, đỉnh=4             | stack=[2, 3, 4]  len=2 best=2
i=7 ')' pop 4, đỉnh=3             | stack=[2, 3]  len=4 best=4
Đáp án: 4


4

### Vì sao Stack đúng? (invariant bằng lời)

Sau khi xử lý xong index `i`, stack luôn có dạng:

```text
[ bức_tường, index các '(' chưa được ghép (tăng dần) ]
```

trong đó `bức_tường` là index của `')'` cuối cùng **không ghép được** (hoặc `-1` nếu chưa có).

- Đoạn hợp lệ **không thể chứa** một `')'` không ghép được (bức tường), cũng **không thể chứa** một `'('` không ghép được (các index đang nằm trong stack). Vậy sau khi pop, `stack[-1]` là ranh giới **gần `i` nhất** mà đoạn không được vượt qua.
- Mọi ký tự nằm giữa `stack[-1]` và `i` đều đã được ghép cặp với nhau → đoạn `s[stack[-1]+1 .. i]` hợp lệ, và đó là đoạn **dài nhất kết thúc tại `i`**.
- Lấy `max` trên mọi `i` → đáp án.

Stack **không bao giờ rỗng** sau mỗi bước (luôn còn bức tường), nên `stack[-1]` luôn an toàn.

**Độ phức tạp:** mỗi index được push tối đa 1 lần và pop tối đa 1 lần → **Time `O(n)`**. Stack có thể chứa tới `n + 1` phần tử (chuỗi toàn `'('`) → **Space `O(n)`**.

## 7. Cách 4 — Hai lượt quét với 2 biến đếm (bộ nhớ `O(1)`)

**Cách cũ vướng ở đâu?** DP và Stack đều tốn `O(n)` bộ nhớ. Quay lại ý tưởng `balance` ở mục 3: ta có thể tìm đáp án chỉ với **hai biến đếm** `left` (số `'('`) và `right` (số `')'`) trong đoạn đang xét không?

### Lượt trái → phải

- Gặp `'('` → `left += 1`; gặp `')'` → `right += 1`.
- Nếu `left == right` → đoạn đang xét hợp lệ, độ dài `2 * right`.
- Nếu `right > left` → có `')'` thừa, đoạn này chết → **reset** `left = right = 0` (bắt đầu đoạn mới từ ký tự sau).

Thử ngay với ví dụ `"(()"` của đề:

In [15]:
def longest_valid_one_pass_only(s: str) -> int:
    left = right = best = 0
    for c in s:
        if c == "(":
            left += 1
        else:
            right += 1
        if left == right:
            best = max(best, 2 * right)
        elif right > left:
            left = right = 0
    return best

for s, expected in [(")()())", 4), ("(()", 2), ("()((())", 4)]:
    print(f"{s!r:10} -> {longest_valid_one_pass_only(s)}  (đúng phải là {expected})")

')()())'   -> 4  (đúng phải là 4)
'(()'      -> 0  (đúng phải là 2)
'()((())'  -> 2  (đúng phải là 4)


### Vì sao một lượt là KHÔNG đủ?

Với `"(()"`: `left` luôn lớn hơn `right` (`1>0`, `2>0`, `2>1`) → **không bao giờ** bằng nhau, và cũng không bao giờ `right > left` để reset. Cặp `"()"` bên trong bị "che" bởi dấu `'('` thừa ở đầu.

Lượt trái→phải chỉ phát hiện được `')'` thừa (bằng reset), còn `'('` thừa thì **im lặng làm hỏng** phép so sánh `left == right`.

**Mấu chốt:** đọc chuỗi theo **chiều ngược lại**, vai trò hai loại ngoặc đổi chỗ: lúc này `'('` thừa sẽ kích hoạt reset (`left > right`). Hai lượt bổ sung cho nhau:

| Lượt | Reset khi | Bắt được đoạn bị "che" bởi |
|---|---|---|
| trái → phải | `right > left` | `')'` thừa |
| phải → trái | `left > right` | `'('` thừa |

Để duyệt ngược dùng `range(n - 1, -1, -1)`: bắt đầu từ `n-1`, bước `-1`, dừng **trước** `-1` (tức là tới 0).

In [16]:
n = 5
print(list(range(n - 1, -1, -1)))   # [4, 3, 2, 1, 0]
s = "(()"
print([s[i] for i in range(len(s) - 1, -1, -1)])

[4, 3, 2, 1, 0]
[')', '(', '(']


In [17]:
def longest_valid_two_pass(s: str) -> int:
    n = len(s)
    best = 0

    left = right = 0                      # Lượt 1: trái -> phải
    for i in range(n):
        if s[i] == "(":
            left += 1
        else:
            right += 1
        if left == right:
            best = max(best, 2 * right)
        elif right > left:                # ')' thừa -> reset
            left = right = 0

    left = right = 0                      # Lượt 2: phải -> trái
    for i in range(n - 1, -1, -1):
        if s[i] == "(":
            left += 1
        else:
            right += 1
        if left == right:
            best = max(best, 2 * left)
        elif left > right:                # '(' thừa -> reset
            left = right = 0

    return best

for s in ["(()", ")()())", "", "()(()", "()((())"]:
    print(f"{s!r:10} -> {longest_valid_two_pass(s)}")

'(()'      -> 2
')()())'   -> 4
''         -> 0
'()(()'    -> 2
'()((())'  -> 4


### Trace hai lượt với `s = "()((())"`

```text
index: 0 1 2 3 4 5 6
s    : ( ) ( ( ( ) )
```

Đáp án đúng là `4` (đoạn `s[3:7] = "(())"`), nhưng nó bị "che" bởi `'('` thừa tại index 2.

**Lượt trái → phải** (reset khi `right > left`):

| i | c | left | right | Ghi chú | best |
|:-:|:-:|:-:|:-:|---|:-:|
| 0 | `(` | 1 | 0 | | 0 |
| 1 | `)` | 1 | 1 | bằng nhau → 2 | 2 |
| 2 | `(` | 2 | 1 | | 2 |
| 3 | `(` | 3 | 1 | | 2 |
| 4 | `(` | 4 | 1 | | 2 |
| 5 | `)` | 4 | 2 | | 2 |
| 6 | `)` | 4 | 3 | không bao giờ bằng nhau! | 2 |

**Lượt phải → trái** (reset khi `left > right`):

| i | c | left | right | Ghi chú | best |
|:-:|:-:|:-:|:-:|---|:-:|
| 6 | `)` | 0 | 1 | | 2 |
| 5 | `)` | 0 | 2 | | 2 |
| 4 | `(` | 1 | 2 | | 2 |
| 3 | `(` | 2 | 2 | bằng nhau → 4 | 4 |
| 2 | `(` | 3 → 0 | 2 → 0 | `'('` thừa → reset | 4 |
| 1 | `)` | 0 | 1 | | 4 |
| 0 | `(` | 1 | 1 | bằng nhau → 2 | 4 |

Kết quả `4`: lượt ngược đã "cứu" đoạn `"(())"`.

In [18]:
def trace_two_pass(s: str) -> int:
    n = len(s)
    best = 0
    for name, order, bad in [("TRÁI -> PHẢI", range(n), "right"),
                             ("PHẢI -> TRÁI", range(n - 1, -1, -1), "left")]:
        print(f"--- {name} (reset khi {bad} lớn hơn) ---")
        left = right = 0
        for i in order:
            if s[i] == "(":
                left += 1
            else:
                right += 1
            note = ""
            if left == right:
                best = max(best, 2 * left)
                note = f"bằng nhau -> {2 * left}"
            elif (right > left) if bad == "right" else (left > right):
                note = f"reset (left={left}, right={right})"
                left = right = 0
            print(f"i={i} {s[i]} | left={left} right={right} | best={best} {note}")
    print("Đáp án:", best)
    return best

trace_two_pass("()((())")

--- TRÁI -> PHẢI (reset khi right lớn hơn) ---
i=0 ( | left=1 right=0 | best=0 
i=1 ) | left=1 right=1 | best=2 bằng nhau -> 2
i=2 ( | left=2 right=1 | best=2 
i=3 ( | left=3 right=1 | best=2 
i=4 ( | left=4 right=1 | best=2 
i=5 ) | left=4 right=2 | best=2 
i=6 ) | left=4 right=3 | best=2 
--- PHẢI -> TRÁI (reset khi left lớn hơn) ---
i=6 ) | left=0 right=1 | best=2 
i=5 ) | left=0 right=2 | best=2 
i=4 ( | left=1 right=2 | best=2 
i=3 ( | left=2 right=2 | best=4 bằng nhau -> 4
i=2 ( | left=0 right=0 | best=4 reset (left=3, right=2)
i=1 ) | left=0 right=1 | best=4 
i=0 ( | left=1 right=1 | best=4 bằng nhau -> 2
Đáp án: 4


4

### Vì sao hai lượt là đủ?

**Không bao giờ ghi nhầm.** Ở lượt 1, khi `left == right`, đoạn đang đếm chưa từng có lúc `right > left` (nếu có thì đã reset) — đúng hai điều kiện của `is_valid` ở mục 3 → đoạn đó hợp lệ thật. Lượt 2 tương tự theo chiều ngược.

**Không bỏ sót.** Gọi `T = s[a..b]` là đoạn hợp lệ dài nhất. Ở lượt 1, khi đọc tới `a`, bộ đếm đang giữ một phần "dính" `P` ngay trước `T` (từ lần reset gần nhất tới `a-1`), và `P` không thừa `')'`.

- Nếu `P` cân bằng (kể cả `P` rỗng) → tới `b` ta có `left == right` → lượt 1 bắt được `T`.
- Nếu `P` thừa `'('` → `s[a-1]` chắc chắn là `'('`. (Nếu là `')'` thì nó ghép được với một `'('` trong `P`, tạo đoạn hợp lệ sát trái `T` → nối dài được `T`, mâu thuẫn với "dài nhất".)

Lập luận y hệt cho lượt 2: hoặc lượt 2 bắt được `T`, hoặc `s[b+1]` chắc chắn là `')'`.
Nếu **cả hai** lượt cùng trượt thì `s[a-1] = '('` và `s[b+1] = ')'` → `"(" + T + ")"` hợp lệ và dài hơn `T` — mâu thuẫn. Vậy ít nhất một lượt bắt được `T`. (Mục 9 kiểm chứng thêm bằng 3000 test ngẫu nhiên.)

**Độ phức tạp:** **Time `O(n)`** (hai vòng lặp), **Space `O(1)`** (chỉ 3 biến).

## 8. Code nộp LeetCode

Bản chính dùng **Stack với mốc `-1`** — ngắn, ít chỉ số, dễ giải thích khi phỏng vấn. (Bản hai lượt quét nằm ở mục Template cuối notebook nếu bạn muốn `O(1)` bộ nhớ.)

In [19]:
class Solution:
    def longestValidParentheses(self, s: str) -> int:
        stack = [-1]
        best = 0
        for i, c in enumerate(s):
            if c == "(":
                stack.append(i)
            else:
                stack.pop()
                if not stack:
                    stack.append(i)
                else:
                    best = max(best, i - stack[-1])
        return best

### Bộ test: ví dụ đề + edge case

| Nhóm | Input | Expected |
|---|---|:-:|
| Ví dụ đề | `"(()"`, `")()())"`, `""` | 2, 4, 0 |
| Một ký tự | `"("`, `")"` | 0, 0 |
| Toàn sai thứ tự | `")("`, `"))(("` | 0, 0 |
| Cả chuỗi hợp lệ | `"()(())"`, `"(()())"` | 6, 6 |
| Ghép hai đoạn cạnh nhau | `"()()"` | 4 |
| Bị cắt bởi `'('` thừa | `"()(()"` | 2 |
| Bị cắt bởi `')'` thừa | `"()())()"` | 4 |
| Đoạn hợp lệ ở cuối | `"(((()"` | 2 |

In [20]:
tests = [
    ("(()", 2), (")()())", 4), ("", 0),
    ("(", 0), (")", 0),
    (")(", 0), ("))((", 0),
    ("()(())", 6), ("(()())", 6),
    ("()()", 4),
    ("()(()", 2),
    ("()())()", 4),
    ("(((()", 2),
]
sol = Solution()
for s, expected in tests:
    got = sol.longestValidParentheses(s)
    assert got == expected, (s, got, expected)
print(f"Tất cả {len(tests)} test PASS!")

big = "()" * 15000                      # n = 30000, chạm giới hạn đề
assert sol.longestValidParentheses(big) == 30000
assert sol.longestValidParentheses("(" * 30000) == 0
print("Test kích thước tối đa PASS!")

Tất cả 13 test PASS!
Test kích thước tối đa PASS!


## 9. Đối chiếu chéo các cách trên input ngẫu nhiên

Lời giải brute force `O(n³)` chậm nhưng **gần như chắc chắn đúng** (đúng theo định nghĩa). Ta dùng nó làm "trọng tài" cho các cách còn lại trên hàng nghìn chuỗi ngẫu nhiên nhỏ. `random.seed(32)` cố định để lần chạy nào cũng ra cùng dữ liệu.

In [21]:
import random
random.seed(32)

methods = {
    "n2": longest_valid_n2,
    "dp": longest_valid_dp,
    "stack": longest_valid_stack,
    "two_pass": longest_valid_two_pass,
    "Solution": Solution().longestValidParentheses,
}
for _ in range(3000):
    length = random.randint(0, 14)
    s = "".join(random.choice("()") for _ in range(length))
    expected = longest_valid_bruteforce(s)
    for name, f in methods.items():
        assert f(s) == expected, (name, s, f(s), expected)
print("3000 chuỗi ngẫu nhiên: mọi cách khớp brute force!")

s = "".join(random.choice("()") for _ in range(30000))
fast = {name: methods[name](s) for name in ["dp", "stack", "two_pass"]}
print("Chuỗi ngẫu nhiên n=30000:", fast)
assert len(set(fast.values())) == 1

3000 chuỗi ngẫu nhiên: mọi cách khớp brute force!
Chuỗi ngẫu nhiên n=30000: {'dp': 12006, 'stack': 12006, 'two_pass': 12006}


## 10. So sánh các cách

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|:-:|:-:|---|
| Brute force | thử mọi `s[i:j]` chẵn + `is_valid` | `O(n³)` | `O(n)` | dễ nghĩ, làm "trọng tài" test |
| Brute force cải tiến | cố định `i`, `balance` chạy theo `j` | `O(n²)` | `O(1)` | vẫn TLE với `n = 3·10⁴` |
| DP | `dp[i]` = dài nhất kết thúc tại `i`, 2 trường hợp | `O(n)` | `O(n)` | luyện tư duy DP, dễ sai chỉ số |
| **Stack + mốc `-1`** | stack index, đáy là "bức tường" | `O(n)` | `O(n)` | **đẹp, nên dùng khi phỏng vấn** |
| Hai lượt quét | `left/right` hai chiều | `O(n)` | `O(1)` | tối ưu bộ nhớ, cần hiểu vì sao cần lượt ngược |

## 11. Lỗi thường gặp

### Lỗi 1: Đếm tổng số cặp ghép được thay vì đoạn **liên tiếp**

Ý tưởng sai: "ghép được bao nhiêu cặp thì đáp án là `2 * số cặp`". Cách này bỏ qua việc các cặp có thể bị **chia cắt** bởi ngoặc thừa.

In [22]:
def wrong_count_pairs(s: str) -> int:
    open_count = pairs = 0
    for c in s:
        if c == "(":
            open_count += 1
        elif open_count > 0:
            open_count -= 1
            pairs += 1
    return 2 * pairs

s = "()(()"
print("SAI  :", wrong_count_pairs(s))       # 4: ghép "()" ở đầu với "()" ở cuối
print("ĐÚNG :", longest_valid_stack(s))     # 2: '(' tại index 2 chia cắt hai cặp

SAI  : 4
ĐÚNG : 2


**Sửa:** phải theo dõi **ranh giới** đoạn (bức tường trong stack, hoặc reset bộ đếm), không chỉ đếm cặp.

### Lỗi 2: Stack quên mốc `-1` hoặc quên push "bức tường mới"

- Không có `-1` ban đầu → chuỗi bắt đầu bằng `"()"` thì sau khi pop, stack rỗng, không có gì để trừ.
- Khi stack rỗng mà chỉ `continue` (không push `i`) → mất ranh giới, đoạn sau sẽ bị tính xuyên qua `')'` thừa.

In [23]:
def wrong_stack_no_new_wall(s: str) -> int:
    stack = [-1]
    best = 0
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        else:
            if len(stack) > 1:            # chỉ pop khi có '(' để ghép
                stack.pop()
                best = max(best, i - stack[-1])
            # SAI: gặp ')' thừa mà không cập nhật bức tường
    return best

s = "())()"
print("SAI  :", wrong_stack_no_new_wall(s))   # 5: tính xuyên qua ')' thừa ở index 2
print("ĐÚNG :", longest_valid_stack(s))       # 2

SAI  : 5
ĐÚNG : 2


**Sửa:** gặp `')'` → luôn `pop()`; nếu stack rỗng → `stack.append(i)` để `')'` này làm bức tường mới.

### Lỗi 3: DP quên cộng `dp[j-1]` ở trường hợp B

Thiếu phần này thì đoạn `"(...)"` mới ghép xong không được nối với đoạn hợp lệ đứng ngay bên trái nó.

In [24]:
def wrong_dp_no_join(s: str) -> int:
    n = len(s)
    dp = [0] * n
    for i in range(1, n):
        if s[i] == ")":
            if s[i - 1] == "(":
                dp[i] = (dp[i - 2] if i >= 2 else 0) + 2
            else:
                j = i - dp[i - 1] - 1
                if j >= 0 and s[j] == "(":
                    dp[i] = dp[i - 1] + 2          # SAI: thiếu + dp[j-1]
    return max(dp, default=0)

s = "()(())"
print("SAI  :", wrong_dp_no_join(s))    # 4: chỉ thấy "(())"
print("ĐÚNG :", longest_valid_dp(s))    # 6: "()" + "(())"

SAI  : 4
ĐÚNG : 6


### Lỗi 4: Chỉ quét một lượt trái → phải

Đã thấy ở mục 7: `"(()"` trả về `0` thay vì `2`. **Sửa:** thêm lượt phải→trái với điều kiện reset **đảo ngược** (`left > right`).

### Lỗi 5: Lặt vặt khác

| Lỗi | Hậu quả | Sửa |
|---|---|---|
| Lượt ngược vẫn reset khi `right > left` | lượt 2 vô dụng, y như lượt 1 | lượt ngược reset khi `left > right` |
| Độ dài tính `i - stack[-1] + 1` | dư 1 | `stack[-1]` là vị trí **trước** đoạn, nên chỉ `i - stack[-1]` |
| Push ký tự `'('` vào stack thay vì index | không tính được độ dài | push `i` |
| Gọi `stack[-1]` khi stack rỗng | `IndexError` | luôn kiểm tra `if not stack` sau khi pop |
| `max(dp)` với `s = ""` | `ValueError: max() arg is an empty sequence` | dùng `max(dp, default=0)` hoặc biến `best` |
| Nhầm substring với subsequence | trả về tổng cặp ghép được | đọc kỹ đề: đoạn **liên tiếp** |

## 12. Mẹo nhận dạng pattern

- Thấy **ngoặc đơn một loại** + "hợp lệ" → nghĩ ngay tới **biến đếm `balance`**; nhiều loại ngoặc (`()[]{}`) → **stack ký tự**.
- Đề hỏi **độ dài / vị trí** của đoạn → stack lưu **index**, không lưu ký tự.
- "Đoạn liên tiếp dài nhất thoả điều kiện" → thử định nghĩa **`dp[i]` = đáp án kết thúc tại `i`**.
- Cần "ranh giới trái" cho phép trừ index → thêm **phần tử mốc (sentinel) `-1`** vào đáy stack. Mẹo này xuất hiện cả ở *Largest Rectangle in Histogram* (#84).
- Muốn bỏ `O(n)` bộ nhớ của một bài quét đếm → xem có quét **hai chiều** để bù trường hợp bị bỏ lỡ không.

## 13. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #20 Valid Parentheses (Easy) | kiểm tra hợp lệ với 3 loại ngoặc → stack ký tự |
| #22 Generate Parentheses (Medium) | sinh mọi chuỗi hợp lệ, dùng điều kiện `left/right` |
| #921 Minimum Add to Make Parentheses Valid (Medium) | biến đếm `balance` + đếm `')'` thừa (Bài tập 5) |
| #1249 Minimum Remove to Make Valid Parentheses (Medium) | stack index để biết ký tự nào cần xoá |
| #678 Valid Parenthesis String (Medium) | thêm ký tự `'*'`, dùng khoảng `[min, max]` của balance |
| #301 Remove Invalid Parentheses (Hard) | BFS/backtracking xoá ít ký tự nhất |
| #84 Largest Rectangle in Histogram (Hard) | stack index + mốc `-1`, cùng công thức `i - stack[-1]` |

### Biến thể nhỏ: trả về chính chuỗi con dài nhất

Chỉ cần nhớ thêm vị trí bắt đầu `stack[-1] + 1` mỗi khi phá kỷ lục.

In [25]:
def longest_valid_substring(s: str) -> str:
    stack = [-1]
    best_len, best_start = 0, 0
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        else:
            stack.pop()
            if not stack:
                stack.append(i)
            elif i - stack[-1] > best_len:
                best_len, best_start = i - stack[-1], stack[-1] + 1
    return s[best_start:best_start + best_len]

for s in [")()())", "())((())", "()(()", ")("]:
    print(f"{s!r:11} -> {longest_valid_substring(s)!r}")

')()())'    -> '()()'
'())((())'  -> '(())'
'()(()'     -> '()'
')('        -> ''


## 14. Bài tập tự luyện

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 15) và **Đáp án** (mục 16, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

Hàm `longest_valid_stack` trả về gì với từng input sau? Với input thứ hai, vì sao **không** phải `4`?

```python
longest_valid_stack(")(")
longest_valid_stack("()(()")
longest_valid_stack("(()())")
longest_valid_stack("))(()")
```

### Bài 2 — Tính tay bảng DP

Lập bảng `dp` (định nghĩa ở mục 5) cho `s = "(()())"`. Ghi rõ mỗi `i` rơi vào nhánh nào (`'('`, A hay B), và ở nhánh B thì `j` bằng bao nhiêu.

### Bài 3 — Điền TODO: Stack với mốc `-1`

Hoàn thiện hàm theo đúng thuật toán mục 6.

In [26]:
def exercise_stack(s: str) -> int:
    stack = [-1]
    best = 0
    for i, c in enumerate(s):
        if c == "(":
            # TODO 1: push index i
            pass
        else:
            # TODO 2: pop đỉnh stack
            # TODO 3: nếu stack rỗng -> push i làm bức tường mới
            # TODO 4: ngược lại -> cập nhật best bằng i - stack[-1]
            pass
    return best

# Tự làm xong rồi mới mở test:
# assert exercise_stack("(()") == 2
# assert exercise_stack(")()())") == 4
# assert exercise_stack("())((())") == 4
# assert exercise_stack("") == 0

### Bài 4 — Tự viết lại: hai lượt quét `O(1)` bộ nhớ

Không nhìn lại mục 7, tự viết hàm hai lượt quét. Nhớ: lượt ngược đổi điều kiện reset.

In [27]:
def exercise_two_pass(s: str) -> int:
    # TODO: lượt trái -> phải (reset khi right > left)
    # TODO: lượt phải -> trái (reset khi left > right)
    raise NotImplementedError

# Gợi ý test:
# assert exercise_two_pass("(()") == 2
# assert exercise_two_pass("()((())") == 4
# assert exercise_two_pass(")()())") == 4
# assert exercise_two_pass("") == 0

### Bài 5 — Biến thể (LeetCode #921): số ngoặc ít nhất cần **thêm** để chuỗi hợp lệ

Được phép chèn `'('` hoặc `')'` vào bất kỳ đâu. Trả về số lần chèn ít nhất.

```python
min_add_to_make_valid("())")    -> 1   # thêm '(' ở đầu
min_add_to_make_valid("(((")    -> 3
min_add_to_make_valid("()")     -> 0
min_add_to_make_valid("()))((") -> 4
```

In [28]:
def min_add_to_make_valid(s: str) -> int:
    # TODO: dùng biến đếm '(' đang mở và biến đếm ')' thừa
    pass

# Gợi ý test:
# assert min_add_to_make_valid("())") == 1
# assert min_add_to_make_valid("(((") == 3
# assert min_add_to_make_valid("()") == 0
# assert min_add_to_make_valid("()))((") == 4

### Bài 6 — Mini challenge: đếm **tất cả** chuỗi con hợp lệ trong `O(n)`

Đếm số cặp `(i, j)` sao cho `s[i:j]` là chuỗi ngoặc hợp lệ khác rỗng.

```python
count_valid_substrings("()()")    -> 3   # "()", "()", "()()"
count_valid_substrings("(())")    -> 2   # "()", "(())"
count_valid_substrings(")()())")  -> 3
count_valid_substrings("((")      -> 0
```

Yêu cầu: `O(n)` thời gian, và tự kiểm tra lại bằng một bản brute force dùng `is_valid`.

In [29]:
def count_valid_substrings(s: str) -> int:
    # TODO: ends[i] = số chuỗi hợp lệ kết thúc tại i
    # TODO: dùng stack index của '(' để tìm đối tác j của ')' tại i
    pass

# Gợi ý test:
# assert count_valid_substrings("()()") == 3
# assert count_valid_substrings("(())") == 2
# assert count_valid_substrings(")()())") == 3
# assert count_valid_substrings("((") == 0

## 15. Gợi ý theo mức

**Bài 1**
- Mức 1: chạy tay thuật toán stack, ghi stack sau mỗi bước.
- Mức 2: với `"()(()"`, để ý `'('` tại index 2 có bao giờ bị pop không.
- Mức 3: `'('` chưa ghép nằm lại trong stack và chặn mọi đoạn đi xuyên qua nó.

**Bài 2**
- Mức 1: `'('` luôn cho `dp = 0`.
- Mức 2: `i = 2` và `i = 4` có `s[i-1] == '('` → nhánh A.
- Mức 3: `i = 5` là nhánh B, `j = 5 - dp[4] - 1`.

**Bài 3**
- Mức 1: chỉ có 4 dòng cần điền, đúng như khung giả mã ở mục 6.
- Mức 2: `stack.append(i)`, `stack.pop()`, `if not stack:`.
- Mức 3: `else: best = max(best, i - stack[-1])`.

**Bài 4**
- Mức 1: hai vòng lặp gần giống hệt nhau, chỉ khác chiều duyệt và điều kiện reset.
- Mức 2: chiều ngược dùng `range(n - 1, -1, -1)`.
- Mức 3: luôn kiểm tra `left == right` trước, rồi mới `elif` điều kiện reset.

**Bài 5**
- Mức 1: giống `is_valid`, nhưng khi `balance` sắp âm thì "chèn một `'('`" thay vì trả `False`.
- Mức 2: biến `open_` đếm `'('` đang mở, biến `add` đếm `')'` không có ai ghép.
- Mức 3: kết quả là `add + open_` (`')'` thừa cần thêm `'('`, `'('` thừa cần thêm `')'`).

**Bài 6**
- Mức 1: giống DP mục 5 — đếm số chuỗi hợp lệ **kết thúc tại `i`** rồi cộng lại.
- Mức 2: nếu `')'` tại `i` ghép với `'('` tại `j` (ghép theo stack), thì `s[j..i]` hợp lệ, và mọi chuỗi hợp lệ khác kết thúc tại `i` là `s[k..j-1] + s[j..i]` với `s[k..j-1]` hợp lệ.
- Mức 3: `ends[i] = 1 + ends[j - 1]` (coi `ends[-1] = 0`), đáp án là `sum(ends)`.

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:** `0`, `2`, `6`, `2`.
Với `"()(()"`: `'('` tại index 2 không bao giờ được đóng, nó nằm lại trong stack làm ranh giới, nên `"()"` ở đầu và `"()"` ở cuối không nối được với nhau. Với `"))(()"`: hai `')'` đầu lần lượt thành bức tường, đoạn hợp lệ duy nhất là `"()"` ở cuối.

**Bài 2:** `s = "(()())"`

| i | s[i] | Nhánh | j | dp[i] |
|:-:|:-:|---|:-:|:-:|
| 0 | `(` | `'('` | – | 0 |
| 1 | `(` | `'('` | – | 0 |
| 2 | `)` | A: `dp[0] + 2` | – | 2 |
| 3 | `(` | `'('` | – | 0 |
| 4 | `)` | A: `dp[2] + 2` | – | 4 |
| 5 | `)` | B: `s[0]=='('`, `dp[4] + 2 + 0` | 0 | 6 |

`dp = [0, 0, 2, 0, 4, 6]`, đáp án `6`. Cell dưới kiểm chứng cả hai bài và chạy đáp án bài 3–6.

In [30]:
# Kiểm chứng Bài 1 và Bài 2
assert [longest_valid_stack(t) for t in [")(", "()(()", "(()())", "))(()"]] == [0, 2, 6, 2]

def answer_dp_table(s: str) -> list:
    n = len(s)
    dp = [0] * n
    for i in range(1, n):
        if s[i] == ")":
            if s[i - 1] == "(":
                dp[i] = (dp[i - 2] if i >= 2 else 0) + 2
            else:
                j = i - dp[i - 1] - 1
                if j >= 0 and s[j] == "(":
                    dp[i] = dp[i - 1] + 2 + (dp[j - 1] if j >= 1 else 0)
    return dp

assert answer_dp_table("(()())") == [0, 0, 2, 0, 4, 6]
print("Bài 1, Bài 2 PASS")

Bài 1, Bài 2 PASS


In [31]:
# Bài 3
def answer_exercise_stack(s: str) -> int:
    stack = [-1]
    best = 0
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        else:
            stack.pop()
            if not stack:
                stack.append(i)
            else:
                best = max(best, i - stack[-1])
    return best


# Bài 4
def answer_exercise_two_pass(s: str) -> int:
    n = len(s)
    best = 0
    left = right = 0
    for i in range(n):
        if s[i] == "(":
            left += 1
        else:
            right += 1
        if left == right:
            best = max(best, 2 * right)
        elif right > left:
            left = right = 0
    left = right = 0
    for i in range(n - 1, -1, -1):
        if s[i] == "(":
            left += 1
        else:
            right += 1
        if left == right:
            best = max(best, 2 * left)
        elif left > right:
            left = right = 0
    return best


assert answer_exercise_stack("(()") == 2
assert answer_exercise_stack(")()())") == 4
assert answer_exercise_stack("())((())") == 4
assert answer_exercise_stack("") == 0
assert answer_exercise_two_pass("(()") == 2
assert answer_exercise_two_pass("()((())") == 4
assert answer_exercise_two_pass(")()())") == 4
assert answer_exercise_two_pass("") == 0
print("Bài 3, Bài 4 PASS")

Bài 3, Bài 4 PASS


In [32]:
# Bài 5
def answer_min_add_to_make_valid(s: str) -> int:
    open_ = 0      # số '(' đang mở chờ đóng
    add = 0        # số '(' phải chèn thêm cho các ')' thừa
    for c in s:
        if c == "(":
            open_ += 1
        elif open_ > 0:
            open_ -= 1
        else:
            add += 1
    return add + open_


# Bài 6
def answer_count_valid_substrings(s: str) -> int:
    n = len(s)
    ends = [0] * n          # ends[i] = số chuỗi hợp lệ kết thúc tại i
    stack = []              # chỉ chứa index của '(' chưa ghép
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        elif stack:
            j = stack.pop()                 # đối tác của ')' tại i
            ends[i] = 1 + (ends[j - 1] if j >= 1 else 0)
    return sum(ends)


def brute_count_valid_substrings(s: str) -> int:
    n = len(s)
    return sum(is_valid(s[i:j]) for i in range(n) for j in range(i + 2, n + 1, 2))


assert answer_min_add_to_make_valid("())") == 1
assert answer_min_add_to_make_valid("(((") == 3
assert answer_min_add_to_make_valid("()") == 0
assert answer_min_add_to_make_valid("()))((") == 4
assert answer_count_valid_substrings("()()") == 3
assert answer_count_valid_substrings("(())") == 2
assert answer_count_valid_substrings(")()())") == 3
assert answer_count_valid_substrings("((") == 0

random.seed(921)
for _ in range(2000):
    t = "".join(random.choice("()") for _ in range(random.randint(0, 12)))
    assert answer_count_valid_substrings(t) == brute_count_valid_substrings(t), t
print("Bài 5, Bài 6 PASS (kể cả 2000 test ngẫu nhiên cho Bài 6)")

Bài 5, Bài 6 PASS (kể cả 2000 test ngẫu nhiên cho Bài 6)


## 17. Template ngắn nên nhớ

**Stack + mốc `-1`** — câu thần chú: *"đáy là bức tường; `')'` thì pop; rỗng thì dựng tường mới; còn thì `i - đỉnh`"*.

```python
class Solution:
    def longestValidParentheses(self, s: str) -> int:
        stack = [-1]
        best = 0
        for i, c in enumerate(s):
            if c == "(":
                stack.append(i)
            else:
                stack.pop()
                if not stack:
                    stack.append(i)
                else:
                    best = max(best, i - stack[-1])
        return best
```

**Hai lượt quét `O(1)` bộ nhớ** — *"bằng nhau thì ghi, thừa phía đang đóng thì reset, rồi làm lại theo chiều ngược"*:

```python
class Solution:
    def longestValidParentheses(self, s: str) -> int:
        best = 0
        for order, opener in ((range(len(s)), "("), (range(len(s) - 1, -1, -1), ")")):
            open_cnt = close_cnt = 0
            for i in order:
                if s[i] == opener:
                    open_cnt += 1
                else:
                    close_cnt += 1
                if open_cnt == close_cnt:
                    best = max(best, 2 * open_cnt)
                elif close_cnt > open_cnt:
                    open_cnt = close_cnt = 0
        return best
```

(Ở lượt ngược, `')'` đóng vai trò "mở" và `'('` đóng vai trò "đóng" — nên một điều kiện reset dùng được cho cả hai lượt.)

In [33]:
# Kiểm chứng bản template hai lượt gọn ở trên
def template_two_pass(s: str) -> int:
    best = 0
    for order, opener in ((range(len(s)), "("), (range(len(s) - 1, -1, -1), ")")):
        open_cnt = close_cnt = 0
        for i in order:
            if s[i] == opener:
                open_cnt += 1
            else:
                close_cnt += 1
            if open_cnt == close_cnt:
                best = max(best, 2 * open_cnt)
            elif close_cnt > open_cnt:
                open_cnt = close_cnt = 0
    return best

random.seed(32)
for _ in range(2000):
    t = "".join(random.choice("()") for _ in range(random.randint(0, 14)))
    assert template_two_pass(t) == longest_valid_bruteforce(t), t
for s, expected in tests:
    assert template_two_pass(s) == expected
print("Template hai lượt PASS")

Template hai lượt PASS


## 18. Checklist trước khi submit

- [ ] Mình trả về **độ dài** (số nguyên), không phải chuỗi con?
- [ ] Đã xử lý `s = ""` (trả `0`, không gọi `max` trên list rỗng)?
- [ ] Stack có khởi tạo mốc `[-1]` chưa? Gặp `')'` có **luôn** `pop()` không?
- [ ] Khi stack rỗng sau `pop()`, đã `append(i)` làm bức tường mới chưa?
- [ ] Độ dài là `i - stack[-1]` (không `+1`)?
- [ ] Nếu dùng DP: trường hợp B có kiểm `j >= 0`, `s[j] == '('` và cộng `dp[j-1]` chưa?
- [ ] Nếu dùng hai lượt: lượt ngược reset khi `left > right` chưa?
- [ ] Đã thử các ca: `"(()"`, `")()())"`, `"()(()"`, `"()(())"`, `"))(("` chưa?
- [ ] Độ phức tạp `O(n)` — an toàn với `n = 3 · 10⁴`.